# Arabic → English pivot: token cost vs accuracy (Experiments 2 & 3)

**Kaggle settings (right panel):** Accelerator = **GPU T4 x2** · Internet = **On** · Persistence = Files.
(GPU + internet need a phone-verified Kaggle account.)

**How to run**
1. *(Optional, recommended)* Run all cells once with `SMOKE = True` (≈10 min) to catch errors while you're awake.
2. Set `SMOKE = False`, then **Save Version → Save & Run All (Commit)**. It runs in the background (≈2–4 h); you can close the browser.
3. When it finishes, open the version → **Output** → download `results.zip` and give it to me.

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate sentencepiece tiktoken

In [ ]:
import os, re, gc, time, json, random, zipfile
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import tiktoken

SMOKE = True                  # <-- set False for the full run
SEED  = 0
MODELS = ["Qwen/Qwen2.5-7B-Instruct", "unsloth/Meta-Llama-3.1-8B-Instruct"]
MT_MODEL = "facebook/nllb-200-distilled-600M"
N_BELEBELE = 20 if SMOKE else 900     # all 900 parallel questions
N_COT      = 10 if SMOKE else 300     # chain-of-thought subset
N_AMMLU    = 40 if SMOKE else 1500    # ArabicMMLU stratified sample
BS_SHORT, BS_LONG = 16, 8
OUT = "/kaggle/working/results"; os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
O200K = tiktoken.get_encoding("o200k_base")          # GPT-4o tokenizer, for API-cost estimates
print(torch.cuda.device_count(), "GPUs |", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Data

In [ ]:
BB = "https://huggingface.co/datasets/facebook/belebele/resolve/main/data/{}.jsonl"
ar = pd.read_json(BB.format("arb_Arab"), lines=True)
en = pd.read_json(BB.format("eng_Latn"), lines=True)
assert len(ar) == len(en) == 900 and (ar.link == en.link).all() and (ar.question_number == en.question_number).all()
bb = pd.DataFrame({"id": range(900), "gold": ar.correct_answer_num.astype(int),
    "p_ar": ar.flores_passage, "q_ar": ar.question, "o_ar": ar[[f"mc_answer{i}" for i in range(1,5)]].values.tolist(),
    "p_en": en.flores_passage, "q_en": en.question, "o_en": en[[f"mc_answer{i}" for i in range(1,5)]].values.tolist()})
bb = bb.sample(n=N_BELEBELE, random_state=SEED).sort_values("id").reset_index(drop=True)

am = pd.read_csv("https://huggingface.co/datasets/MBZUAI/ArabicMMLU/resolve/main/All/test.csv")
am = am[am.get("is_few_shot", 0) == 0].copy()
am["gold"] = am["Answer Key"].map({"A":1,"B":2,"C":3,"D":4,"E":5})
am = am.dropna(subset=["gold", "Question", "Option 1", "Option 2"])
am = (am.groupby("Group", group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), N_AMMLU // am.Group.nunique()), random_state=SEED))
        .reset_index(drop=True))
am["o_ar"] = am[[f"Option {i}" for i in range(1,6)]].apply(lambda r: [str(x) for x in r if pd.notna(x)], axis=1)
am["ctx_ar"] = am["Context"].fillna("").astype(str)
print("Belebele:", len(bb), "| ArabicMMLU:", len(am), am.Group.value_counts().to_dict())

## 2. Local machine translation Arabic → English (NLLB, zero paid tokens)

In [ ]:
SENT = re.compile(r'(?<=[\.\!\?\u061F])\s+')
mt_tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="arb_Arab")
mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=torch.float16).to("cuda:0").eval()
ENG = mt_tok.convert_tokens_to_ids("eng_Latn")

def _mt_batch(texts, bs=48):
    out = [None]*len(texts); order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    for k in range(0, len(order), bs):
        idx = order[k:k+bs]; batch = [texts[i] for i in idx]
        enc = mt_tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=400).to("cuda:0")
        with torch.no_grad():
            g = mt.generate(**enc, forced_bos_token_id=ENG, num_beams=2,
                            max_new_tokens=int(enc.input_ids.shape[1]*1.6)+16)
        for i, t in zip(idx, mt_tok.batch_decode(g, skip_special_tokens=True)): out[i] = t
    return out

def translate_all(strings):
    # sentence-split long texts, translate unique sentences once, re-join
    uniq = sorted(set(s for s in strings if s))
    pieces = {s: [x for x in SENT.split(s) if x.strip()] for s in uniq}
    sents = sorted(set(x for v in pieces.values() for x in v))
    torch.cuda.synchronize(); t0 = time.time()
    tr = dict(zip(sents, _mt_batch(sents)))
    torch.cuda.synchronize(); dt = time.time() - t0
    return {s: " ".join(tr[x] for x in pieces[s]) for s in uniq} | {"": ""}, dt, len(sents)

strings = list(bb.p_ar) + list(bb.q_ar) + [o for os_ in bb.o_ar for o in os_] \
        + list(am.Question.astype(str)) + list(am.ctx_ar) + [o for os_ in am.o_ar for o in os_]
MT, mt_seconds, n_sent = translate_all(strings)
bb["p_mt"] = bb.p_ar.map(MT); bb["q_mt"] = bb.q_ar.map(MT); bb["o_mt"] = bb.o_ar.map(lambda l: [MT[o] for o in l])
am["q_mt"] = am.Question.astype(str).map(MT); am["ctx_mt"] = am.ctx_ar.map(MT); am["o_mt"] = am.o_ar.map(lambda l: [MT[o] for o in l])
json.dump({"mt_seconds": mt_seconds, "n_sentences": n_sent, "n_strings": len(set(strings))}, open(f"{OUT}/mt_timing.json","w"))
bb.to_parquet(f"{OUT}/belebele_with_mt.parquet"); am.to_parquet(f"{OUT}/ammlu_with_mt.parquet")
print(f"translated {n_sent} sentences in {mt_seconds:.0f}s"); print(bb.q_ar[0], "->", bb.q_mt[0], "| human:", bb.q_en[0])
del mt; gc.collect(); torch.cuda.empty_cache()

## 3. Prompts

In [ ]:
def opts(o): return "\n".join(f"{i+1}. {x}" for i, x in enumerate(o))
def p_en(passage, q, o, cot=False, ctx_label="Passage"):
    head = f"{ctx_label}: {passage}\n\n" if passage else ""
    tail = ("Think step by step briefly, then write the final line as 'Answer: N'." if cot
            else "Reply with the number of the correct option only.")
    return f"Answer the question by choosing the number of the correct option.\n\n{head}Question: {q}\n{opts(o)}\n\n{tail}"
def p_ar(passage, q, o, cot=False):
    head = f"النص: {passage}\n\n" if passage else ""
    tail = ("فكّر خطوة بخطوة باختصار باللغة العربية، ثم اكتب في السطر الأخير: 'الإجابة: N'." if cot
            else "أجب برقم الخيار الصحيح فقط.")
    return f"أجب عن السؤال باختيار رقم الخيار الصحيح.\n\n{head}السؤال: {q}\n{opts(o)}\n\n{tail}"
def p_translate(block):
    return ("Translate the following Arabic text into English. Keep the numbered options. "
            "Output only the translation.\n\n" + block)
def p_answer_translated(block, cot=False):
    tail = ("Think step by step briefly, then write the final line as 'Answer: N'." if cot
            else "Reply with the number of the correct option only.")
    return f"Answer the question by choosing the number of the correct option.\n\n{block}\n\n{tail}"

AR_DIG = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
def parse(text, k, cot):
    t = (text or "").translate(AR_DIG)
    if cot:
        m = re.findall(r'(?:Answer|الإجابة|الاجابة)\s*[:：]?\s*\(?([1-9])', t)
        if m and 1 <= int(m[-1]) <= k: return int(m[-1])
        d = [int(x) for x in re.findall(r'\b([1-9])\b', t) if 1 <= int(x) <= k]
        return d[-1] if d else -1
    d = re.findall(r'([1-9])', t)
    d = [int(x) for x in d if 1 <= int(x) <= k]
    return d[0] if d else -1

## 4. LLM runner (batched, timed, token-counted)

In [ ]:
def load(repo):
    tok = AutoTokenizer.from_pretrained(repo, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.float16, device_map="auto").eval()
    for a in ("temperature", "top_p", "top_k"): setattr(m.generation_config, a, None)
    return tok, m

def run(tok, m, prompts, max_new, bs):
    texts = [tok.apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True) for p in prompts]
    order = sorted(range(len(texts)), key=lambda i: -len(texts[i]))
    res = [None]*len(texts)
    for k in range(0, len(order), bs):
        idx = order[k:k+bs]
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.no_grad():
            out = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
        torch.cuda.synchronize(); dt = time.time() - t0
        gen = out[:, enc.input_ids.shape[1]:]
        for j, i in enumerate(idx):
            g = gen[j]; stop = (g == tok.pad_token_id) | (g == tok.eos_token_id)
            n_gen = int(stop.nonzero()[0, 0]) if stop.any() else int(g.shape[0])
            res[i] = dict(prompt_tokens=int(enc.attention_mask[j].sum()), gen_tokens=n_gen,
                          output=tok.decode(g[:n_gen], skip_special_tokens=True),
                          batch_seconds=dt, batch_size=len(idx))
    return res

## 5. Experiments

In [ ]:
import traceback
ERRLOG = f"{OUT}/errors.txt"
def safe(name, fn):
    try:
        fn(); return True
    except Exception as e:
        with open(ERRLOG, "a", encoding="utf-8") as f:
            f.write(f"=== {name}\n{traceback.format_exc()}\n")
        print("FAILED:", name, "->", type(e).__name__, str(e)[:200], "(logged, continuing)")
        gc.collect(); torch.cuda.empty_cache(); return False

def save(rows, name):
    df = pd.DataFrame(rows); df.to_parquet(f"{OUT}/{name}.parquet"); return df

def belebele_runs(tok, m, mshort):
    for cot in (False, True):
        mode = "cot" if cot else "direct"
        sub = bb if not cot else bb.head(N_COT)
        mx, bs = (512, BS_LONG) if cot else (16, BS_SHORT)
        jobs = {"en_human": [p_en(r.p_en, r.q_en, r.o_en, cot) for r in sub.itertuples()],
                "ar_direct": [p_ar(r.p_ar, r.q_ar, r.o_ar, cot) for r in sub.itertuples()],
                "ar_mt_pivot": [p_en(r.p_mt, r.q_mt, r.o_mt, cot) for r in sub.itertuples()]}
        for strat, prompts in jobs.items():
            name = f"{mshort}__belebele__{strat}__{mode}"
            if os.path.exists(f"{OUT}/{name}.parquet"): continue
            def job(prompts=prompts, strat=strat, name=name, sub=sub, mx=mx, bs=bs, cot=cot, mode=mode):
                res = run(tok, m, prompts, mx, bs)
                save([dict(model=mshort, dataset="belebele", strategy=strat, mode=mode, id=r.id, gold=r.gold,
                           pred=parse(x["output"], 4, cot), prompt=p, **x) for r, p, x in zip(sub.itertuples(), prompts, res)], name)
                print(name, "done")
            safe(name, job)
        # self-translate: the LLM translates the Arabic block itself (2 calls, both counted)
        name = f"{mshort}__belebele__ar_self_translate__{mode}"
        def st_job(name=name, sub=sub, mx=mx, bs=bs, cot=cot, mode=mode):
            blocks = [f"Passage: {r.p_ar}\n\nQuestion: {r.q_ar}\n{opts(r.o_ar)}" for r in sub.itertuples()]
            t1 = run(tok, m, [p_translate(b) for b in blocks], 768, BS_LONG)
            p2 = [p_answer_translated(x["output"], cot) for x in t1]
            t2 = run(tok, m, p2, mx, bs)
            save([dict(model=mshort, dataset="belebele", strategy="ar_self_translate", mode=mode, id=r.id, gold=r.gold,
                       pred=parse(b["output"], 4, cot), prompt=p, output=b["output"],
                       prompt_tokens=a["prompt_tokens"] + b["prompt_tokens"], gen_tokens=a["gen_tokens"] + b["gen_tokens"],
                       translate_prompt_tokens=a["prompt_tokens"], translate_gen_tokens=a["gen_tokens"], translation=a["output"],
                       batch_seconds=a["batch_seconds"]/a["batch_size"] + b["batch_seconds"]/b["batch_size"], batch_size=1)
                  for r, p, a, b in zip(sub.itertuples(), p2, t1, t2)], name)
            print(name, "done")
        if not os.path.exists(f"{OUT}/{name}.parquet"): safe(name, st_job)

def ammlu_runs(tok, m, mshort):
    jobs = {"ar_direct": [p_ar(r.ctx_ar, r.Question, r.o_ar) for r in am.itertuples()],
            "ar_mt_pivot": [p_en(r.ctx_mt, r.q_mt, r.o_mt, ctx_label="Context") for r in am.itertuples()]}
    for strat, prompts in jobs.items():
        name = f"{mshort}__ammlu__{strat}__direct"
        if os.path.exists(f"{OUT}/{name}.parquet"): continue
        def job(prompts=prompts, strat=strat, name=name):
          res = run(tok, m, prompts, 16, BS_SHORT)
          save([dict(model=mshort, dataset="ammlu", strategy=strat, mode="direct", id=r.ID, gold=int(r.gold),
                     group=r.Group, subject=r.Subject, n_opts=len(r.o_ar), pred=parse(x["output"], len(r.o_ar), False),
                     prompt=p, **x) for r, p, x in zip(am.itertuples(), prompts, res)], name)
          print(name, "done")
        safe(name, job)

for repo in MODELS:
    mshort = repo.split("/")[-1]; t0 = time.time(); tok = m = None
    def model_job():
        global tok, m
        tok, m = load(repo)
        belebele_runs(tok, m, mshort); ammlu_runs(tok, m, mshort)
    safe(f"MODEL {mshort}", model_job)
    print(f"== {mshort} finished in {(time.time()-t0)/60:.1f} min")
    m = None; gc.collect(); torch.cuda.empty_cache()

## 6. Quick look + package results

In [ ]:
parts = [f for f in os.listdir(OUT) if f.count("__") == 3]
if parts:
  df = pd.concat([pd.read_parquet(f"{OUT}/{f}") for f in parts], ignore_index=True)
  df["correct"] = (df.pred == df.gold).astype(int)
  df["o200k_prompt"] = df.prompt.map(lambda s: len(O200K.encode(s)))
  print(df.groupby(["dataset","model","mode","strategy"]).agg(
        n=("correct","size"), acc=("correct","mean"), prompt_tok=("prompt_tokens","mean"),
        gen_tok=("gen_tokens","mean"), unparsed=("pred", lambda s: (s == -1).mean())).round(3).to_string())
  df.to_csv(f"{OUT}/all_results.csv", index=False)
else:
  print('no result files produced - see errors.txt')
with zipfile.ZipFile("/kaggle/working/results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT): z.write(f"{OUT}/{f}", f)
print("results.zip ready:", os.path.getsize("/kaggle/working/results.zip")//1024, "KB")